In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from evaluation import Evaluator, save_evaluation_results
from pathlib import Path


root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

nn_Unet_Evals = []
SAM_Evals = []
Recontour_evals = []


['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [2]:
for subject_nr in range(len(subjects)):
    data = DataLoader(parentfolder=root,subject_nr=subject_nr,volume_of_interest="CTVT",verbose=True)
    unc_handler = UG_prompter(data=data)
    seg_handler = Segmentation(data=data)

    unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, method=method, mode="mean")
    unc_handler.compute_band_thickness(method=method)   

    prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=4.0,
        interpix_dist=3,
        pixel_interval=10,
        angle_step=15,
        method=method)
    
    bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=0.0)
    
    seg_handler.load_dense_prompt()
    seg_handler.add_prompt_dict(prompt_dict=prompts)
    seg_handler.check_loaded_prompts()
    seg_handler.construct_prompts()
    #seg_handler.run_segmentation(propagation_style=propagation_style, nr_propagation_slices=1)
    seg_handler.run_segmentation_sets(propagation_style=propagation_style, nr_propagation_slices=5, weighting_strategy="custom", weighting_list=[0.0,0.0,0.2,0.8], threshold=0.0, bbox_prompts_by_slice=bbox_prompts)
    seg_handler.remove_distant_slices(tolerance_frames=0)

    eval_handler = Evaluator(segmentation=seg_handler)
    metrics = eval_handler.compute_all(surface_dice_tol=1.0)

    SAM_Evals.append(metrics)


Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


there are 52 many pixels
there are 142 many pixels
there are 184 many pixels
there are 208 many pixels
there are 228 many pixels
there are 251 many pixels
there are 277 many pixels
there are 294 many pixels
there are 304 many pixels
there are 306 many pixels
there are 302 many pixels
there are 295 many pixels
there are 285 many pixels
there are 269 many pixels
there are 247 many pixels
there are 215 many pixels
there are 182 many pixels
there are 132 many pixels
iter=00 | thr=0.194962 | band=1.86 mm | error=1.14
there are 52 many pixels
there are 142 many pixels
there are 184 many pixels
there are 208 many pixels
there are 228 many pixels
there are 251 many pixels
there are 277 many pixels
there are 294 many pixels
there are 304 many pixels
there are 306 many pixels
there are 302 many pixels
there are 295 many pixels
there are 285 many pixels
there are 269 many pixels
there are 247 many pixels
there are 215 many pixels
there are 182 many pixels
there are 132 many pixels
iter=01 | thr=0

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (from middle slice 36)...


...

Running segmentation for prompt set 'bbox' with slices: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Adding prompt(s) on slice 45
Adding prompt(s) on slice 46
Adding prompt(s) on slice 47
Adding prompt(s) on slice 48
Adding prompt(s) on slice 49
Adding prompt(s) on slice 50
Adding prompt(s) on slice 51
Forward propagation (from middle slice 42)...


propagate in video: 100%|██████████| 46/46 [00:11<00:00,  3.94it/s]


Backward propagation (from middle slice 42)...


propagate in video: 100%|██████████| 43/43 [00:10<00:00,  4.14it/s]


Kept slices 33 to 51. Removed predicted segmentation outside dense mask ±0 slices.


In [3]:
save_evaluation_results(SAM_Evals, "results", filename="weighted_multiprop_optimtest",)

Saved CSV to:   results\weighted_multiprop_optimtest.csv
Saved Excel to: results\weighted_multiprop_optimtest.xlsx


,subject name,HD_mm,HD95_mm,MSD_mm,ASSD_mm,SurfaceDice@1.0mm,CentroidDistance_mm,PredictionVolume_mm3,GroundTruthVolume_mm3,AbsVolumeDifference_mm3,RelativeVolumeDifference_percent
0,newAcq_050f229dc2bdb64c,7.500000,4.721595,1.153947,1.174926,0.594132,1.473827,53821.418494,56456.502133,2635.083639,-4.667458
1,newAcq_0b4940fa31a1d650,19.156395,11.289668,4.207670,2.892172,0.292960,3.778122,58439.957499,71034.074924,12594.117425,-17.729685
2,newAcq_0cc559a8bd82a14a,5.000000,3.281600,1.231345,1.141560,0.565329,0.762730,96372.305152,84551.790263,11820.514889,13.980207
3,newAcq_1b911d6cb2348f30,6.156085,3.017790,0.862452,1.025495,0.597300,0.981514,39596.582086,47480.954525,7884.372439,-16.605337
4,newAcq_1e0f8b9b01ce5f0b,6.579922,4.106871,1.348657,1.291315,0.535252,1.805177,44226.109764,41013.021957,3213.087807,7.834311
5,newAcq_250d6075dd465a1a,10.000000,5.000000,1.557759,1.424502,0.528824,1.908795,103720.430379,105251.701876,1531.271497,-1.454866
6,newAcq_433a8d44fddd5b7f,5.435622,2.586416,0.989888,0.942172,0.619391,1.067690,37304.345026,34946.175931,2358.169095,6.748003
7,newAcq_47ceabdbca398517,6.423647,2.651933,0.896504,0.777258,0.710358,1.078484,32713.277702,29735.347485,2977.930217,10.014782
8,newAcq_486b7494ee9d71e7,3.528470,1.992000,0.462937,0.466682,0.832940,0.595534,32854.949375,32363.901463,491.047912,1.517270
9,newAcq_4a136e8fe320bd13,10.566361,5.381071,1.961910,1.878477,0.414801,2.706057,57133.953786,53211.547176,3922.406609,7.371345
